# 🔬 Agent Nanoscribe V2 - Interactive Explorer

This notebook allows you to run the **Named Object Agent** directly from a prompt and visualize the resulting 3D geometry and manufacturing data.

### How to use:
1. Run the **Setup** cell to load dependencies.
2. Enter your design intent in the **Prompt** cell.
3. Run the **Execute Pipeline** cell to generate the design.
4. View the **Visualizations** at the bottom.

In [ ]:
import os
import sys
import json
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, Image, JSON

# Ensure we are in the right directory and add src to path
current_dir = Path(os.getcwd())
src_dir = current_dir / "src"
sys.path.insert(0, str(src_dir))

from NamedObjectAgent import run_design

print(f"✅ Environment Ready")
print(f"Source directory: {src_dir}")

## ✍️ Enter Your Design Prompt

In [ ]:
# EDIT THIS PROMPT
user_prompt = "A hierarchical metamaterial composed of a 3x3 grid of 'micro-flowers'. Each flower has a 10um tall central stem (cylinder) and 4 tilted petals (boxes) arranged around it."
category = "Exploration"

print(f"Target Prompt: {user_prompt}")

## ⚙️ Execute Pipeline
This will call the LLM to design the structure, reduce it to primitives, enrich it with manufacturing parameters, and generate renders.

In [ ]:
print("🚀 Running Agent Nanoscribe V2 Pipeline...")
try:
    result = run_design(user_prompt, category)
    
    output_path = Path(result['output_path'])
    print(f"\n✨ SUCCESS")
    print(f"Job Name:      {result['job_name']}")
    print(f"Total Primitives: {result['num_primitives']}")
    print(f"Output Folder:   {output_path.relative_to(current_dir)}")
    
    if result['errors']:
        print(f"\n⚠️ Warnings: {result['errors']}")

except Exception as e:
    print(f"\n❌ ERROR: {str(e)}")
    import traceback
    traceback.print_exc()

## 📊 Design Structure (High-Level)
This shows how the LLM organized your geometry into named, reusable objects.

In [ ]:
design_file = output_path / "design.json"
with open(design_file) as f:
    design_data = json.load(f)

print("Object Library Keys:", list(design_data.get('objects', {}).keys()))
display(JSON(design_data))

## 🖼️ Visualizations
The system automatically generates top and side views of the full assembly.

In [ ]:
render_dir = output_path / "Renders" / "final_assembly"

views = ["top.png", "side_xz.png", "side_yz.png"]
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

for i, view in enumerate(views):
    img_path = render_dir / view
    if img_path.exists():
        img = mpimg.imread(img_path)
        axes[i].imshow(img)
        axes[i].set_title(view.replace('.png', '').upper())
        axes[i].axis('off')
    else:
        axes[i].text(0.5, 0.5, f"Missing: {view}", ha='center')
        axes[i].axis('off')

plt.tight_layout()
plt.show()

## 🛠️ Manufacturing Data
Summary of the risk enrichment performed by the Manufacturing Agent.

In [ ]:
enriched_file = output_path / "enriched_reduced.json"
with open(enriched_file) as f:
    enriched_data = json.load(f)

prims = enriched_data.get('primitives', [])
if prims:
    sample = prims[0]
    print(f"Sample Primitive Risk: {sample.get('risk', 'N/A')}")
    print(f"Local Parameters: {sample.get('local_parameters', 'N/A')}")
    
    risks = [p.get('risk', 0) for p in prims]
    plt.figure(figsize=(8, 4))
    plt.hist(risks, bins=20, color='orange', alpha=0.7)
    plt.title("Distribution of Geometric Risk Across Primitives")
    plt.xlabel("Risk Score [0-1]")
    plt.ylabel("Count")
    plt.show()